# Enhanced DICOM inspection

The `.dcm` in `401-CSF_QF_retrospectivetrigger` is a single **Enhanced MR Image** (multi-frame) file, not a classic one-file-per-slice series. Per-frame `ImageType` lives in `PerFrameFunctionalGroupsSequence`, not the top-level `ImageType` tag.

In [ ]:
from pathlib import Path
import pydicom

folder_path = Path("/mnt/cina/GLIMPSE_OSA/data/sourcedata/dicom/GLOtwo001/v2/401-CSF_QF_retrospectivetrigger")
dcm_path = next(folder_path.glob("*.dcm"))

ds = pydicom.dcmread(dcm_path, stop_before_pixels=True)
print("File:", dcm_path.name)
print("SOP Class UID:", ds.SOPClassUID)
print("Top-level ImageType:", getattr(ds, "ImageType", None))
print("NumberOfFrames:", getattr(ds, "NumberOfFrames", None))

## Enumerate all per-frame image types

Two places to look:
- Standard `MR Image Frame Type Sequence` (0018,9226) -> `Frame Type` (0008,9007)
- Philips private stack (2005,140f) -> `Image Type` (0008,0008) (more specific, vendor-style codes like `M_FFE`, `M_PCA`, `PHASE CONTRAST M`)

In [ ]:
pf = ds.PerFrameFunctionalGroupsSequence

std_types = {}
priv_types = {}

for frame in pf:
    ft_seq = frame.get((0x0018, 0x9226))
    if ft_seq:
        ft = tuple(ft_seq[0].FrameType)
        std_types[ft] = std_types.get(ft, 0) + 1

    priv = frame.get((0x2005, 0x140f))
    if priv:
        it = tuple(priv[0][(0x0008, 0x0008)].value)
        priv_types[it] = priv_types.get(it, 0) + 1

print(f"{len(pf)} total frames\n")

print("Standard MR Image Frame Type values (count):")
for t, n in sorted(std_types.items()):
    print(f"  {n:3d}  {t}")

print("\nPhilips private per-frame Image Type values (count):")
for t, n in sorted(priv_types.items()):
    print(f"  {n:3d}  {t}")